# Baseline v2: Chat Template + input_layernorm

Fixes two issues from `02_baseline_reproduction.ipynb`:

1. **Chat template:** The paper wraps all inputs through `apply_chat_template()` before tokenizing.
   Plain text gets wrapped as `[{"role": "user", "content": "..."}]`. This matters because
   Llama-Instruct was trained with special tokens for role formatting.

2. **Hook point:** The paper hooks into `input_layernorm` (the residual stream entering layer N),
   not the full layer output. This captures what layers 0..N-1 have built up, before layer N
   transforms it.

## v2b fixes (matching paper exactly)
- `fit_intercept=False` in LogisticRegression (paper does this, redundant with StandardScaler centering)
- `max_length=8192` (paper uses 2**13; was 512, truncating multi-turn Anthropic/ToolACE dialogues)

## Cache Naming
Cache files use `v2b_` prefix to distinguish from the original v2 run (which used max_length=512).

---

## Part 0: Setup

In [ ]:
# Install dependencies if not in a pre-configured environment
# Lambda Labs / local venv: run setup_lambda.sh first (installs everything)
# Colab: install what's missing
import importlib
missing = [pkg for pkg in ["bitsandbytes", "dotenv", "accelerate"] if not importlib.util.find_spec(pkg)]
if missing:
    %pip install -U bitsandbytes python-dotenv accelerate transformers scikit-learn tqdm

In [ ]:
import sys
import json
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# add experiments/ to path so we can import lib/
sys.path.insert(0, str(Path("..").resolve()))

from lib.env import (
    detect_environment, resolve_base_dir, setup_paths, get_device,
    get_gpu_vram_gb, recommend_batch_size, free_gpu_memory, setup_hf_auth,
    download_from_colab, list_cache,
)
from lib.data import ensure_datasets, get_dataset_paths, load_dataset
from lib.model import load_model
from lib.activations import get_activations_cached
from lib.probe import LinearProbe
from lib.evaluation import evaluate_probe, analyze_errors

# ---------------------------------------------------------------------------
# Environment, paths, device
# ---------------------------------------------------------------------------
ENV      = detect_environment()
BASE_DIR = resolve_base_dir()
DEVICE   = get_device()
VRAM_GB  = get_gpu_vram_gb()
paths    = setup_paths(BASE_DIR)

DATA_DIR     = paths["data_dir"]
CACHE_DIR    = paths["cache_dir"]
CACHE_PREFIX = "v2b"

print(f"Environment:  {ENV}")
print(f"Base dir:     {BASE_DIR}")
print(f"Device:       {DEVICE}")
print(f"VRAM:         {VRAM_GB:.1f} GB")
print(f"Cache prefix: {CACHE_PREFIX}")

In [ ]:
# Authenticate with HuggingFace (reads HF_TOKEN from .env or Colab Secrets)
setup_hf_auth()

In [ ]:
# Download datasets if needed, then load
DATASET_PATHS = ensure_datasets(DATA_DIR)

train_data     = load_dataset(DATASET_PATHS["train"])
test_data      = load_dataset(DATASET_PATHS["test"])
anthropic_test = load_dataset(DATASET_PATHS["anthropic_test"])
toolace_test   = load_dataset(DATASET_PATHS["toolace_test"])

print(f"\nDataset sizes:")
print(f"  Train:          {len(train_data):>5} ({sum(e.label for e in train_data)} high-stakes)")
print(f"  Test (synth):   {len(test_data):>5} ({sum(e.label for e in test_data)} high-stakes)")
print(f"  Anthropic HH:   {len(anthropic_test):>5} ({sum(e.label for e in anthropic_test)} high-stakes)")
print(f"  ToolACE:        {len(toolace_test):>5} ({sum(e.label for e in toolace_test)} high-stakes)")

In [ ]:
# Inspect: show raw messages vs what chat template produces
print("=" * 60)
print("TRAINING EXAMPLE (plain text -> wrapped as user message):")
print("=" * 60)
ex = next(e for e in train_data if e.label == 1)
print(f"Messages: {ex.messages}")

print("\n" + "=" * 60)
print("ANTHROPIC EXAMPLE (already dialogue):")
print("=" * 60)
ex = anthropic_test[0]
for m in ex.messages[:3]:
    print(f"  [{m['role']}] {m['content'][:100]}...")

print("\n" + "=" * 60)
print("TOOLACE EXAMPLE (already dialogue):")
print("=" * 60)
ex = toolace_test[0]
for m in ex.messages[:3]:
    print(f"  [{m['role']}] {m['content'][:100]}...")

---
## Part 2: Load Model

In [ ]:
# VRAM-aware: quantize only if VRAM < 20 GB (e.g. T4 16GB)
# A10 (24GB), A100 (40/80GB) run fp16 for cleaner activations
model, tokenizer = load_model()

In [ ]:
# Verify chat template works
sample_msgs = [{"role": "user", "content": "Hello, is this a test?"}]
formatted = tokenizer.apply_chat_template(sample_msgs, tokenize=False, add_generation_prompt=False)
print("Chat template output:")
print(repr(formatted[:300]))

In [ ]:
# Show what the model actually sees after chat template
for name, ex in [("Training", train_data[0]), ("Anthropic", anthropic_test[0])]:
    formatted = tokenizer.apply_chat_template(ex.messages, tokenize=False)
    print(f"\n{name} -> chat template output:")
    print(formatted[:300])

---
## Part 3: Activation Extraction

### Key changes from v1:
1. **Chat template** applied to messages before tokenizing
2. **Hook on `input_layernorm`** instead of the full layer output
3. **Strip BOS token** after tokenization (matching the paper's `v[:, 1:]`)

In [ ]:
LAYER_IDX  = 16
BATCH_SIZE = recommend_batch_size(VRAM_GB)

print(f"Extracting activations from layer {LAYER_IDX} (input_layernorm, with chat template)...")
print(f"VRAM: {VRAM_GB:.1f} GB -> batch_size: {BATCH_SIZE}")

# Training set (single-turn, short)
X_train = get_activations_cached(
    model, tokenizer, train_data, LAYER_IDX,
    cache_name="train", cache_dir=CACHE_DIR,
    cache_prefix=CACHE_PREFIX, batch_size=BATCH_SIZE,
)
y_train = np.array([e.label for e in train_data])

# Synthetic test set (single-turn, short)
X_test = get_activations_cached(
    model, tokenizer, test_data, LAYER_IDX,
    cache_name="test_synthetic", cache_dir=CACHE_DIR,
    cache_prefix=CACHE_PREFIX, batch_size=BATCH_SIZE,
)
y_test = np.array([e.label for e in test_data])

print(f"\nActivation shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")

free_gpu_memory()

In [ ]:
free_gpu_memory()

X_anthropic = get_activations_cached(
    model, tokenizer, anthropic_test, LAYER_IDX,
    cache_name="anthropic_test", cache_dir=CACHE_DIR,
    cache_prefix=CACHE_PREFIX, batch_size=BATCH_SIZE,
)
y_anthropic = np.array([e.label for e in anthropic_test])

print(f"\nEval activation shapes:")
print(f"  Anthropic: {X_anthropic.shape}")

In [ ]:
free_gpu_memory()

X_toolace = get_activations_cached(
    model, tokenizer, toolace_test, LAYER_IDX,
    cache_name="toolace_test", cache_dir=CACHE_DIR,
    cache_prefix=CACHE_PREFIX, batch_size=BATCH_SIZE,
)
y_toolace = np.array([e.label for e in toolace_test])

print(f"\nEval activation shapes:")
print(f"  ToolACE:   {X_toolace.shape}")

---
## Part 4: Train Probe

In [ ]:
free_gpu_memory()

print("Training linear probe...")
probe = LinearProbe(C=1e-3)
probe.fit(X_train, y_train)

probe_path = CACHE_DIR / f"{CACHE_PREFIX}_probe_layer{LAYER_IDX}.pkl"
probe.save(probe_path)
print(f"Saved probe to {probe_path.name}")

---
## Part 5: Evaluate

In [ ]:
print("=" * 60)
print("EVALUATION RESULTS (v2b: chat template + input_layernorm)")
print("=" * 60)

results = {}
results["synthetic_test"] = evaluate_probe(probe, X_test,      y_test,      "Synthetic Test")
results["anthropic"]      = evaluate_probe(probe, X_anthropic, y_anthropic, "Anthropic HH")
results["toolace"]        = evaluate_probe(probe, X_toolace,   y_toolace,   "ToolACE")

# Save results
results_path = CACHE_DIR / f"{CACHE_PREFIX}_baseline_results.json"
artifacts = {
    "version":    "v2b",
    "changes":    ["chat_template", "input_layernorm", "strip_bos", "fit_intercept_false", "max_length_8192"],
    "model_name": "meta-llama/Llama-3.1-8B-Instruct",
    "layer_idx":  LAYER_IDX,
    "results":    results,
    "train_size": len(train_data),
    "hidden_dim": model.config.hidden_size,
    "vram_gb":    VRAM_GB,
    "env":        ENV,
}
with open(results_path, "w") as f:
    json.dump(artifacts, f, indent=2)
print(f"\nResults saved to: {results_path.name}")

download_from_colab(CACHE_DIR, results_path.name, CACHE_PREFIX)

In [ ]:
# ROC curves
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

eval_sets = [
    ("Synthetic Test", X_test,      y_test),
    ("Anthropic HH",  X_anthropic, y_anthropic),
    ("ToolACE",       X_toolace,   y_toolace),
]

for ax, (name, X, y) in zip(axes, eval_sets):
    probs = probe.predict_proba(X)
    fpr, tpr, _ = roc_curve(y, probs)
    auroc = roc_auc_score(y, probs)

    ax.plot(fpr, tpr, 'b-', lw=2, label=f'AUROC = {auroc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(name)
    ax.legend(loc='lower right')
    ax.grid(alpha=0.3)

fig.suptitle('v2b: Chat Template + input_layernorm + max_length=8192', fontsize=12, fontweight='bold')
plt.tight_layout()

roc_path = CACHE_DIR / f"{CACHE_PREFIX}_roc_curves.png"
plt.savefig(roc_path, dpi=150)
plt.show()

download_from_colab(CACHE_DIR, roc_path.name, CACHE_PREFIX)

In [ ]:
# Comparison: v1 vs v2
v1_results_path = CACHE_DIR / "baseline_results.json"

print("\n" + "=" * 70)
print("COMPARISON: v1 (raw text, layer output) vs v2 (chat template, layernorm)")
print("=" * 70)
print(f"{'Dataset':<20} {'v1 AUROC':<12} {'v2 AUROC':<12} {'Delta':<10}")
print("-" * 55)

if v1_results_path.exists():
    with open(v1_results_path) as f:
        v1 = json.load(f)
    v1_results = v1["results"]
    for name in results:
        v1_auroc = v1_results.get(name, {}).get("auroc", float("nan"))
        v2_auroc = results[name]["auroc"]
        delta    = v2_auroc - v1_auroc
        print(f"{name:<20} {v1_auroc:<12.4f} {v2_auroc:<12.4f} {delta:+.4f}")

    v1_mean = np.mean([v1_results[n]["auroc"] for n in results if n in v1_results])
    v2_mean = np.mean([results[n]["auroc"] for n in results])
    print("-" * 55)
    print(f"{'Mean':<20} {v1_mean:<12.4f} {v2_mean:<12.4f} {v2_mean - v1_mean:+.4f}")
    print(f"\nPaper target: ~0.91 mean AUROC")
else:
    print("v1 results not found. Run 02_baseline_reproduction.ipynb first.")
    print(f"\nv2 results:")
    for name, m in results.items():
        print(f"  {name:<20} {m['auroc']:.4f}")
    v2_mean = np.mean([m['auroc'] for m in results.values()])
    print(f"  {'Mean':<20} {v2_mean:.4f}")
    print(f"  Paper target: ~0.91")

---
## Part 6: Error Analysis

In [ ]:
analyze_errors(probe, X_test,      test_data,      "Synthetic Test")
analyze_errors(probe, X_anthropic, anthropic_test, "Anthropic HH")
analyze_errors(probe, X_toolace,   toolace_test,   "ToolACE")

---
## Part 7: Save Artifacts

In [ ]:
# Final summary of all cached artifacts
print("All v2b artifacts:")
list_cache(CACHE_DIR, prefix=CACHE_PREFIX)

# Download everything as a zip (Colab only)
download_from_colab(CACHE_DIR, cache_prefix=CACHE_PREFIX)

In [ ]:
# Upload cache from local (Colab only, for resuming)
# from lib.env import upload_to_colab
# upload_to_colab(CACHE_DIR)

---
## Summary

### Changes from v1
| Aspect | v1 (notebook 02) | v2 (this notebook) |
|--------|-----------------|--------------------|  
| Input formatting | Raw text / `ROLE: content` | `apply_chat_template()` |
| Hook point | `model.layers[16]` (layer output) | `model.layers[16].input_layernorm` |
| BOS token | Kept | Stripped (matching paper) |
| Data structure | `Example.text` (string) | `Example.messages` (chat format) |

### Results
See comparison table above.

### Next Steps
1. If v2 matches paper: proceed to Phase 2 (Indonesian translation)
2. If v2 still off: investigate regularization sweep, layer sweep, or other differences